# Silver2 Data Cleaning Notebook

This notebook now builds Silver datasets from Bronze Delta splits under `data/bronze/` instead of reading raw CSV files directly.

## Scope
- Input: Bronze Delta tables in `data/bronze/train`, `data/bronze/test`, `data/bronze/validation`
- Goal: detect, document, and fix data quality issues for each split
- Corrupt-row handling: use `_corrupt_record` for structural repair when Bronze parsing failed
- Output: cleaned split dataframes + exported split files under `data/silver/{train,test,validation}`

## 1) Imports and Paths

In [44]:
import csv
import html
import json
import re
import unicodedata
from pathlib import Path
from collections import Counter

import duckdb
import pandas as pd

PROJECT_ROOT = Path('../../').resolve()
RAW_DIR = PROJECT_ROOT / 'reviews (copy)'
BRONZE_DIR = PROJECT_ROOT / 'data' / 'bronze'
SILVER_DIR = PROJECT_ROOT / 'data' / 'silver'

BRONZE_SPLIT_PATHS = {
    'train': BRONZE_DIR / 'train',
    'test': BRONZE_DIR / 'test',
    'validation': BRONZE_DIR / 'validation',
}

SILVER_SPLIT_DIRS = {
    'train': SILVER_DIR / 'train',
    'test': SILVER_DIR / 'test',
    'validation': SILVER_DIR / 'validation',
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Bronze dir:    {BRONZE_DIR}')
for split, path in BRONZE_SPLIT_PATHS.items():
    print(f' - {split:<10} {path} (exists={path.exists()})')

Project root: C:\Users\shara\Documents\MasterIS\BD\BigDataProject
Bronze dir:    C:\Users\shara\Documents\MasterIS\BD\BigDataProject\data\bronze
 - train      C:\Users\shara\Documents\MasterIS\BD\BigDataProject\data\bronze\train (exists=True)
 - test       C:\Users\shara\Documents\MasterIS\BD\BigDataProject\data\bronze\test (exists=True)
 - validation C:\Users\shara\Documents\MasterIS\BD\BigDataProject\data\bronze\validation (exists=True)


## 2) Reference Data for Domain Validation

In [45]:
with open(RAW_DIR / 'marketplace.json', 'r', encoding='utf-8') as f:
    marketplace_raw = json.load(f)

with open(RAW_DIR / 'category.json', 'r', encoding='utf-8') as f:
    category_raw = json.load(f)

# marketplace.json is column-oriented ({col: {idx: value}}), while
# category.json is row-oriented ([{id:..., name:...}, ...]).
# This normalization makes both structures consistent.
if isinstance(marketplace_raw, dict):
    marketplace_ref = pd.DataFrame(marketplace_raw)
else:
    marketplace_ref = pd.DataFrame(marketplace_raw)

if isinstance(category_raw, dict):
    category_ref = pd.DataFrame(category_raw)
else:
    category_ref = pd.DataFrame(category_raw)

valid_marketplace_ids = set(pd.to_numeric(marketplace_ref['id'], errors='coerce').dropna().astype(int).tolist())
valid_category_ids = set(pd.to_numeric(category_ref['id'], errors='coerce').dropna().astype(int).tolist())

print('Valid marketplace IDs:', sorted(valid_marketplace_ids))
print('Valid category IDs:   ', sorted(valid_category_ids)[:20], '...')
print('Marketplace reference rows:', len(marketplace_ref))
print('Category reference rows:', len(category_ref))

Valid marketplace IDs: [0, 1, 2, 3]
Valid category IDs:    [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19] ...
Marketplace reference rows: 4
Category reference rows: 29


## 3) Cleaning Rules and Repair Functions

In [46]:
# Ordered list of expected fields when reconstructing a malformed raw record from _corrupt_record.
EXPECTED_FIELDS = [
    'Row_id',
    'product_id',
    'product_parent',
    'product_title',
    'vine',
    'verified_purchase',
    'review_headline',
    'review_body',
    'review_date',
    'marketplace_id',
    'product_category_id',
    'label',
]

# Only these three columns contain free-text requiring HTML cleaning and normalization.
TEXT_FIELDS = ['product_title', 'review_headline', 'review_body']

### 3a) Text Cleaning (`clean_text`)

Free-text fields (`product_title`, `review_headline`, `review_body`) contain several content-level errors found in this dataset:

| Error type | Example in data | Fix applied |
|---|---|---|
| HTML entities | `&amp;`, `&eacute;`, `&#34;` | `html.unescape()` |
| HTML tags | `<br />`, `<br>` | Replaced with a space; remaining tags stripped |
| Diacritics/accents | `Wáyné`, `yớú`, `árrớgánt`, `pớpúlár` | Conditional: remove for UK only (marketplace_id=1); preserve for other markets |
| Excess whitespace | double spaces, leading/trailing | `re.sub(r'\s+', ' ', ...).strip()` |

The function returns a `(value, was_changed)` tuple. `was_changed=True` causes a `text_cleaned_<col>` issue to be logged, making it easy to count how many rows were affected by each fix.

**How conditional diacritic removal works:**
- For UK reviews (marketplace_id=1): diacritics are removed (e.g., `café` → `cafe`)
  1. Use `unicodedata.normalize('NFD', ...)` to decompose accented characters into base + combining marks (e.g., `á` → `a` + combining acute)
  2. Filter out all combining marks (Unicode category `Mn`)
  3. Result: all accented characters become their base ASCII form
- For other marketplaces: diacritics are preserved to maintain linguistic correctness (e.g., French `café`, Spanish `niño`)

In [47]:
def clean_text(text, marketplace_id=None):
    """
    Clean a single text value through five stages:
      1. HTML entity decoding  — e.g. &amp; → &,  &eacute; → é,  &#34; → "
      2. HTML tag removal      — <br /> becomes a space; all other tags are stripped
      3. Diacritic removal     — NFD decompose then strip combining marks (only for UK; marketplace_id=1)
      4. Unicode normalization — NFKC normalizes remaining compatibility characters
      5. Whitespace collapse   — consecutive spaces/newlines reduced to a single space

    Args:
        text: The text value to clean
        marketplace_id: If 1 (UK), remove diacritics; otherwise preserve them for multilingual correctness

    Returns (cleaned_value, was_changed).
    Empty strings are returned as (None, True) so callers can treat them as missing.
    """
    if text is None:
        return None, False

    original = str(text)
    value = original.strip()
    if value == '':
        return None, original != ''

    value = html.unescape(value)
    value = re.sub(r'<br\s*/?>', ' ', value, flags=re.IGNORECASE)
    value = re.sub(r'<[^>]+>', ' ', value)
    
    # Remove diacritics only for UK marketplace (marketplace_id=1); preserve for other markets
    if marketplace_id == 1:
        value = unicodedata.normalize('NFD', value)
        value = ''.join(c for c in value if unicodedata.category(c) != 'Mn')
    
    value = unicodedata.normalize('NFKC', value)
    value = re.sub(r'\s+', ' ', value).strip()

    if value == '':
        return None, True

    return value, value != original

### 3b) Categorical Field Normalization (`normalize_yn`, `normalize_label`)

Three fields have a constrained set of valid values:
- `vine` and `verified_purchase` must be `Y` or `N`
- `label` must be a boolean (`True` / `False`)

In practice the raw data contains variants such as `"Yes"`, `"TRUE"`, `"1"`, and `"False"`. Both functions map all known variants to the canonical form and return `None` for anything unrecognized, which gets flagged in the issue log.


In [48]:
def normalize_yn(value):
    """Map Y/N field variants to canonical 'Y' or 'N'. Returns None for unrecognized values."""
    if value is None:
        return None
    v = str(value).strip().upper()
    if v in {'Y', 'YES', 'TRUE', 'T', '1'}:
        return 'Y'
    if v in {'N', 'NO', 'FALSE', 'F', '0'}:
        return 'N'
    return None


def normalize_label(value):
    """Map label variants to Python bool. Returns None for unrecognized values."""
    if value is None:
        return None
    v = str(value).strip().lower()
    if v in {'true', '1', 't', 'yes', 'y'}:
        return True
    if v in {'false', '0', 'f', 'no', 'n'}:
        return False
    return None


### 3c) Row Parsing and Structural Repair (`parse_row_with_repair`)

This is the main entry point called for every raw CSV line. It handles structural issues in order, then delegates to the cleaning and normalization helpers above.

**Structural error categories handled:**

1. **Field count mismatch - too many fields** - unquoted commas inside free-text columns (`review_headline`, `review_body`) cause spurious CSV splits, producing more fields than expected. Repaired using an anchor heuristic: lock stable left/right fields, then merge middle tokens back as headline/body.
2. **Too few fields** - not enough data to reconstruct a valid row; immediately rejected.

After structural repair, it applies type coercion, date validation, domain checks, and categorical normalization. All anomalies are collected in an `issues` list so every problem in a row is traceable.

**Rejection conditions (row is dropped entirely):**
- Unparseable raw line (CSV exception)
- Too few fields

**Issue-flagged but kept (stored as `None`):**
- Invalid or missing `review_date`
- Missing `product_id`
- Unknown `marketplace_id` or `product_category_id` (relative to reference files)
- Invalid `vine`, `verified_purchase`, or `label` values

In [49]:
def apply_quality_rules(parsed, expect_label=True):
    """Apply type normalization, domain checks, and text cleaning to a parsed record."""
    record = dict(parsed)
    issues = []

    record['Row_id'] = (record.get('Row_id') or '').strip() or None
    if record['Row_id'] is None:
        issues.append('missing_row_id')

    # Parse marketplace_id early so we can decide on diacritic handling for text.
    marketplace_id_raw = record.get('marketplace_id')
    marketplace_id_numeric = pd.to_numeric(marketplace_id_raw, errors='coerce')
    marketplace_id_for_cleaning = int(marketplace_id_numeric) if not pd.isna(marketplace_id_numeric) else None

    for col in TEXT_FIELDS:
        cleaned, changed = clean_text(record.get(col), marketplace_id=marketplace_id_for_cleaning)
        record[col] = cleaned
        if changed:
            issues.append(f'text_cleaned_{col}')

    record['product_id'] = (record.get('product_id') or '').strip() or None
    record['product_parent'] = pd.to_numeric(record.get('product_parent'), errors='coerce')
    record['marketplace_id'] = marketplace_id_numeric
    record['product_category_id'] = pd.to_numeric(record.get('product_category_id'), errors='coerce')

    record['review_date'] = pd.to_datetime(record.get('review_date'), errors='coerce')
    if pd.isna(record['review_date']):
        issues.append('invalid_or_missing_review_date')
        record['review_date'] = None
    else:
        record['review_date'] = record['review_date'].date()

    record['vine'] = normalize_yn(record.get('vine'))
    record['verified_purchase'] = normalize_yn(record.get('verified_purchase'))

    if expect_label:
        record['label'] = normalize_label(record.get('label'))
    else:
        # Label is intentionally absent in test/validation.
        record['label'] = None

    if record['product_id'] is None:
        issues.append('missing_product_id')

    if pd.isna(record['marketplace_id']):
        issues.append('marketplace_id_not_numeric')
        record['marketplace_id'] = None
    else:
        record['marketplace_id'] = int(record['marketplace_id'])
        if record['marketplace_id'] not in valid_marketplace_ids:
            issues.append('marketplace_id_unknown')

    if pd.isna(record['product_category_id']):
        issues.append('product_category_id_not_numeric')
        record['product_category_id'] = None
    else:
        record['product_category_id'] = int(record['product_category_id'])
        if record['product_category_id'] not in valid_category_ids:
            issues.append('product_category_id_unknown')

    if pd.isna(record['product_parent']):
        issues.append('product_parent_not_numeric')
        record['product_parent'] = None
    else:
        record['product_parent'] = int(record['product_parent'])

    if record['vine'] is None:
        issues.append('vine_invalid_or_missing')
    if record['verified_purchase'] is None:
        issues.append('verified_purchase_invalid_or_missing')
    if expect_label and record['label'] is None:
        issues.append('label_invalid_or_missing')

    return record, issues


def parse_row_with_repair(raw_record):
    if raw_record is None or str(raw_record).strip() == '':
        return None, ['empty_raw_row'], None

    try:
        fields = next(csv.reader([raw_record]))
    except Exception:
        return None, ['csv_parse_exception'], None

    fields = [f.strip() if f is not None else None for f in fields]
    issues = []

    strategy = 'direct'

    if len(fields) == len(EXPECTED_FIELDS):
        parsed = dict(zip(EXPECTED_FIELDS, fields))
    elif len(fields) > len(EXPECTED_FIELDS):
        # Reconstruct around stable anchors for malformed rows split by unquoted commas.
        left = fields[:6]
        right = fields[-4:]
        middle = fields[6:-4]
        parsed = {
            'Row_id': left[0],
            'product_id': left[1],
            'product_parent': left[2],
            'product_title': left[3],
            'vine': left[4],
            'verified_purchase': left[5],
            'review_headline': middle[0] if len(middle) > 0 else None,
            'review_body': ','.join(middle[1:]) if len(middle) > 1 else None,
            'review_date': right[0],
            'marketplace_id': right[1],
            'product_category_id': right[2],
            'label': right[3],
        }
        strategy = 'repaired'
        issues.append('field_count_mismatch_repaired')
    else:
        return None, ['too_few_fields_unrecoverable'], None

    parsed, quality_issues = apply_quality_rules(parsed, expect_label=True)
    issues.extend(quality_issues)

    return parsed, issues, strategy

## 4) Load Bronze Splits and Build Clean Candidate Datasets

In [50]:
def load_bronze_split(split_name):
    split_path = BRONZE_SPLIT_PATHS[split_name]
    if not split_path.exists():
        raise FileNotFoundError(f'Bronze split not found: {split_path}')

    query = f"SELECT * FROM delta_scan('{split_path.as_posix()}')"
    with duckdb.connect(database=':memory:') as con:
        return con.execute(query).df()


split_results = {}
issue_counter = Counter()
total_raw_rows = 0

for split_name in ['train', 'test', 'validation']:
    bronze_df = load_bronze_split(split_name)
    accepted_rows = []
    rejected_rows = []

    for row_idx, row in bronze_df.iterrows():
        total_raw_rows += 1
        row_data = row.to_dict()
        source_file = str(row_data.get('_source_file') or '')
        line_number = row_data.get('_index')

        corrupt_record = row_data.get('_corrupt_record')
        has_corrupt_record = isinstance(corrupt_record, str) and corrupt_record.strip() != ''

        if has_corrupt_record:
            parsed, issues, strategy = parse_row_with_repair(corrupt_record)

            if parsed is None:
                # Train can keep strict rejection; test/validation must keep all rows for submission.
                if split_name == 'train':
                    for issue in issues:
                        issue_counter[issue] += 1
                    rejected_rows.append({
                        'dataset_split': split_name,
                        '_source_file': source_file,
                        '_line_number': line_number,
                        'rejection_reasons': ';'.join(issues),
                        'raw_record': str(corrupt_record)[:1000],
                    })
                    continue

                parsed = {
                    'Row_id': row_data.get('Row_id'),
                    'product_id': row_data.get('product_id'),
                    'product_parent': row_data.get('product_parent'),
                    'product_title': row_data.get('product_title'),
                    'vine': row_data.get('vine'),
                    'verified_purchase': row_data.get('verified_purchase'),
                    'review_headline': row_data.get('review_headline'),
                    'review_body': row_data.get('review_body'),
                    'review_date': row_data.get('review_date'),
                    'marketplace_id': row_data.get('marketplace_id'),
                    'product_category_id': row_data.get('product_category_id'),
                    'label': row_data.get('label'),
                }
                parsed, fallback_issues = apply_quality_rules(parsed, expect_label=False)
                issues = list(set(issues + fallback_issues + ['corrupt_record_unrepaired_kept']))
                strategy = 'bronze_clean_from_corrupt_record_kept'
            else:
                issues.append('from_corrupt_record')
                strategy = f'{strategy}_from_corrupt_record'
        else:
            parsed = {
                'Row_id': row_data.get('Row_id'),
                'product_id': row_data.get('product_id'),
                'product_parent': row_data.get('product_parent'),
                'product_title': row_data.get('product_title'),
                'vine': row_data.get('vine'),
                'verified_purchase': row_data.get('verified_purchase'),
                'review_headline': row_data.get('review_headline'),
                'review_body': row_data.get('review_body'),
                'review_date': row_data.get('review_date'),
                'marketplace_id': row_data.get('marketplace_id'),
                'product_category_id': row_data.get('product_category_id'),
                'label': row_data.get('label'),
            }
            parsed, issues = apply_quality_rules(parsed, expect_label=(split_name == 'train'))
            strategy = 'bronze_clean'

        for issue in issues:
            issue_counter[issue] += 1

        parsed['dataset_split'] = split_name
        parsed['_source_file'] = source_file
        parsed['_line_number'] = line_number if line_number is not None else (row_idx + 1)
        parsed['record_origin'] = strategy
        parsed['detected_issues'] = ';'.join(sorted(set(issues))) if issues else ''
        accepted_rows.append(parsed)

    cleaned_df_split = pd.DataFrame(accepted_rows)
    rejected_df_split = pd.DataFrame(rejected_rows)
    split_results[split_name] = {
        'bronze_rows': len(bronze_df),
        'cleaned_df': cleaned_df_split,
        'rejected_df': rejected_df_split,
    }

cleaned_df = pd.concat([split_results[s]['cleaned_df'] for s in split_results], ignore_index=True)
rejected_df = pd.concat([split_results[s]['rejected_df'] for s in split_results], ignore_index=True)

print('Total raw rows across splits:', total_raw_rows)
for split_name in ['train', 'test', 'validation']:
    result = split_results[split_name]
    repaired_count = int((result['cleaned_df']['record_origin'].str.contains('repaired', na=False)).sum()) if len(result['cleaned_df']) else 0
    print(f"{split_name:<10} bronze={result['bronze_rows']:<8} accepted={len(result['cleaned_df']):<8} rejected={len(result['rejected_df']):<8} repaired={repaired_count}")

Total raw rows across splits: 12000
train      bronze=9614     accepted=9614     rejected=0        repaired=0
test       bronze=1137     accepted=1137     rejected=0        repaired=0
validation bronze=1249     accepted=1249     rejected=0        repaired=0


## 5) Deduplicate and Finalize Silver-Ready Dataset

### Deduplication Strategy

Deduplication removes rows that are **exact duplicates** based on the core review content:

```
.drop_duplicates(subset=['product_id', 'review_date', 'review_body'], keep='first')
```

**When a row is dropped:**
- Two or more rows have the *same* `product_id`, `review_date`, AND `review_body`
- These three columns together form a natural key: they uniquely identify a review
- If all three match, the rows are considered identical reviews (same product, same day, same text)
- `keep='first'` means: keep the first occurrence (by sort order), drop all subsequent duplicates

**Why these three columns?**
- `product_id` — identifies which product was reviewed
- `review_date` — when the review was written
- `review_body` — the actual review text (prevents near-duplicates; exact text match only)

**Example:**
If a review appears in two different train files (e.g., `train-1.csv` and `train-2.csv`), or the same file was ingested twice, the deduplication step will catch it and keep only one copy.

**Note:** Rows with missing `review_date` or `product_id` are *not* deduplicated by this logic (since `None != None` in pandas). They are treated as unique rows. This is intentional — if we can't form a complete key, we can't reliably say "this is a duplicate."


In [51]:
silver_splits = {}
duplicates_removed_by_split = {}

for split_name in ['train', 'test', 'validation']:
    cleaned_split = split_results[split_name]['cleaned_df']
    if cleaned_split.empty:
        silver_split = cleaned_split.copy()
        duplicates_removed = 0
    elif split_name == 'train':
        # Deduplicate train only; keep test/validation row counts unchanged for submission.
        before = len(cleaned_split)
        silver_split = (
            cleaned_split
            .sort_values(['_source_file', '_line_number'])
            .drop_duplicates(subset=['product_id', 'review_date', 'review_body'], keep='first')
            .reset_index(drop=True)
        )
        duplicates_removed = before - len(silver_split)
    else:
        silver_split = cleaned_split.reset_index(drop=True)
        duplicates_removed = 0

    # Submission rules for held-out splits: no label column and normalized missing text markers.
    if split_name in ['test', 'validation']:
        if 'label' in silver_split.columns:
            silver_split = silver_split.drop(columns=['label'])

        object_cols = silver_split.select_dtypes(include=['object', 'string']).columns
        if len(object_cols) > 0:
            silver_split.loc[:, object_cols] = silver_split.loc[:, object_cols].replace(
                to_replace=r'(?i)^\s*(nan|none|null)\s*$',
                value=None,
                regex=True,
            )

    if duplicates_removed > 0:
        issue_counter['duplicate_rows_removed'] += duplicates_removed

    silver_split['_ingested_at'] = pd.Timestamp.now('UTC')
    silver_splits[split_name] = silver_split
    duplicates_removed_by_split[split_name] = duplicates_removed

silver_df = pd.concat([silver_splits[s] for s in silver_splits], ignore_index=True)
duplicates_removed = sum(duplicates_removed_by_split.values())

print('Rows after split finalization:')
for split_name in ['train', 'test', 'validation']:
    dedup_applied = 'yes' if split_name == 'train' else 'no'
    print(f"{split_name:<10} rows={len(silver_splits[split_name]):<8} duplicates_removed={duplicates_removed_by_split[split_name]} dedup_applied={dedup_applied}")
print('Total rows after finalization:', len(silver_df))
print('Total duplicates removed:    ', duplicates_removed)

Rows after split finalization:
train      rows=9613     duplicates_removed=1 dedup_applied=yes
test       rows=1137     duplicates_removed=0 dedup_applied=no
validation rows=1249     duplicates_removed=0 dedup_applied=no
Total rows after finalization: 11999
Total duplicates removed:     1


## 6) Missing Values Profiling (Train Only)

In [52]:
# Profile missing values on train only (test/validation are kept intact for submission).
train_missing_df = silver_splits.get('train', pd.DataFrame())

missing_profile = pd.DataFrame({
    'column': train_missing_df.columns,
    'dtype': train_missing_df.dtypes.astype(str).values,
    'missing_count': train_missing_df.isna().sum().values,
    'missing_pct': (train_missing_df.isna().mean().values * 100).round(2)
}).sort_values(['missing_count', 'missing_pct'], ascending=False)

print('Train columns with missing values:')
display(missing_profile[missing_profile['missing_count'] > 0])

missing_by_type = (
    missing_profile.assign(
        type_family=missing_profile['dtype'].map(
            lambda d: 'numeric' if d.startswith(('int', 'float')) else (
                'datetime' if d.startswith('datetime') else (
                    'boolean' if d in ('bool', 'boolean') else 'text_or_category'
                )
            )
        )
    )
    .groupby('type_family', dropna=False)
    .agg(
        columns=('column', 'count'),
        total_missing=('missing_count', 'sum'),
        avg_missing_pct=('missing_pct', 'mean')
    )
    .reset_index()
    .sort_values('total_missing', ascending=False)
)

Train columns with missing values:


,column,dtype,missing_count,missing_pct
6,review_headline,str,79,0.82
3,product_title,str,7,0.07
8,review_date,object,1,0.01


## 7) Data Quality Findings (Errors Detected and Fixes Applied)

In [57]:
dq_summary = pd.DataFrame([
    {'issue': k, 'count': v, 'pct_of_raw_rows': round(100.0 * v / max(total_raw_rows, 1), 2)}
    for k, v in issue_counter.items()
]).sort_values(['count', 'issue'], ascending=[False, True]).reset_index(drop=True)

coverage = pd.DataFrame([{
    'raw_rows': total_raw_rows,
    'accepted_before_dedup': len(cleaned_df),
    'accepted_after_dedup': len(silver_df),
    'rejected_rows': len(rejected_df),
    'duplicates_removed': duplicates_removed,
    'coverage_pct': round(100.0 * len(silver_df) / max(total_raw_rows, 1), 2),
}])

# Build split-level issue counts from detected_issues field.
split_issue_rows = (
    silver_df[['dataset_split', 'detected_issues']]
    .assign(issue=lambda d: d['detected_issues'].str.split(';'))
    .explode('issue')
)
split_issue_rows = split_issue_rows[split_issue_rows['issue'].notna() & (split_issue_rows['issue'] != '')]

raw_rows_by_split = pd.Series({
    split: split_results[split]['bronze_rows']
    for split in ['train', 'test', 'validation']
}, name='raw_rows')

dq_summary_by_split = (
    split_issue_rows.groupby(['dataset_split', 'issue'], dropna=False)
    .size()
    .reset_index(name='count')
    .merge(raw_rows_by_split.rename_axis('dataset_split').reset_index(), on='dataset_split', how='left')
    .assign(pct_of_split_raw_rows=lambda d: (100.0 * d['count'] / d['raw_rows'].clip(lower=1)).round(2))
    .sort_values(['dataset_split', 'count', 'issue'], ascending=[True, False, True])
    .reset_index(drop=True)
)

print('Coverage summary:')
display(coverage)

print('Issue summary (top 30, all splits combined):')
display(dq_summary.head(30))

print('Issue summary grouped by dataset split (top 15 per split):')
display(
    dq_summary_by_split.groupby('dataset_split', group_keys=False)
    .head(15)
    .reset_index(drop=True)
)

Coverage summary:


,raw_rows,accepted_before_dedup,accepted_after_dedup,rejected_rows,duplicates_removed,coverage_pct
0,12000,12000,11999,0,1,99.99


Issue summary (top 30, all splits combined):


,issue,count,pct_of_raw_rows
0,text_cleaned_review_body,6213,51.77
1,from_corrupt_record,1051,8.76
2,text_cleaned_review_headline,318,2.65
3,invalid_or_missing_review_date,270,2.25
4,corrupt_record_unrepaired_kept,269,2.24
5,marketplace_id_not_numeric,269,2.24
6,too_few_fields_unrecoverable,269,2.24
7,product_category_id_not_numeric,240,2.00
8,text_cleaned_product_title,225,1.88
9,vine_invalid_or_missing,2,0.02


Issue summary grouped by dataset split (top 15 per split):


,dataset_split,issue,count,raw_rows,pct_of_split_raw_rows
0,test,text_cleaned_review_body,559,1137,49.16
1,test,corrupt_record_unrepaired_kept,122,1137,10.73
2,test,invalid_or_missing_review_date,122,1137,10.73
3,test,marketplace_id_not_numeric,122,1137,10.73
4,test,too_few_fields_unrecoverable,122,1137,10.73
5,test,product_category_id_not_numeric,107,1137,9.41
6,test,text_cleaned_review_headline,34,1137,2.99
7,test,text_cleaned_product_title,17,1137,1.50
8,test,verified_purchase_invalid_or_missing,1,1137,0.09
9,test,vine_invalid_or_missing,1,1137,0.09


## 8) Final Validation (Pandas Only)

In [54]:
train_only = silver_splits.get('train', pd.DataFrame())

final_validation = pd.DataFrame([{
    'rows_after_cleaning': len(silver_df),
    'rows_with_detected_issues': int((silver_df['detected_issues'] != '').sum()) if len(silver_df) else 0,
    'repaired_rows': int((silver_df['record_origin'].str.contains('repaired', na=False)).sum()) if len(silver_df) else 0,
    'null_product_id': int(silver_df['product_id'].isna().sum()) if len(silver_df) else 0,
    'null_review_date': int(silver_df['review_date'].isna().sum()) if len(silver_df) else 0,
    'invalid_vine_after_normalization': int(silver_df['vine'].isna().sum()) if len(silver_df) else 0,
    'invalid_verified_purchase_after_normalization': int(silver_df['verified_purchase'].isna().sum()) if len(silver_df) else 0,
    'invalid_label_after_normalization_train_only': int(train_only['label'].isna().sum()) if len(train_only) else 0,
}])

display(final_validation)

origin_breakdown = (
    silver_df.groupby(['dataset_split', 'record_origin'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
)
display(origin_breakdown)

,rows_after_cleaning,rows_with_detected_issues,repaired_rows,null_product_id,null_review_date,invalid_vine_after_normalization,invalid_verified_purchase_after_normalization,invalid_label_after_normalization_train_only
0,11999,6615,0,0,270,2,1,0


,dataset_split,record_origin,rows
2,train,bronze_clean,8562
4,validation,bronze_clean,1102
3,train,direct_from_corrupt_record,1051
0,test,bronze_clean,1015
5,validation,bronze_clean_from_corrupt_record_kept,147
1,test,bronze_clean_from_corrupt_record_kept,122


In [55]:
silver_df

,Row_id,product_id,product_parent,product_title,vine,verified_purchase,review_headline,review_body,review_date,marketplace_id,product_category_id,label,dataset_split,_source_file,_line_number,record_origin,detected_issues,_ingested_at
0,9,B001N2MZT8,903886718,Green Zone [DVD],N,Y,green zone,I found at first it was a little difficult to ...,2010-11-15,1.0,3.0,False,train,file:///c:/Users/shara/Documents/MasterIS/BD/B...,42949672960,bronze_clean,,2026-03-11 14:16:27.519050+00:00
1,11,B00GCBVE0Q,282740618,Le secret de Green Knowe,N,Y,nan,J'ai aimé cette histoire. Les acteurs - et sur...,2014-11-23,2.0,3.0,False,train,file:///c:/Users/shara/Documents/MasterIS/BD/B...,42949672961,bronze_clean,,2026-03-11 14:16:27.519050+00:00
2,19,1423165691,883799517,A Disney Sketchbook.,N,N,okay mais...,est-ce une coincidence que la plupart des prin...,2012-12-22,0.0,0.0,False,train,file:///c:/Users/shara/Documents/MasterIS/BD/B...,42949672962,bronze_clean,,2026-03-11 14:16:27.519050+00:00
3,33,0061091480,623343977,Your Erroneous Zones,N,N,Arrogant,Wáyné Dyér is á pớpúlár áméricán pérsớnál grớw...,2009-07-21,0.0,0.0,True,train,file:///c:/Users/shara/Documents/MasterIS/BD/B...,42949672963,bronze_clean,,2026-03-11 14:16:27.519050+00:00
4,34,B00HZ4CYOY,647510225,König der Mathematik Junior,N,Y,Tớllé Máthé Ápp...,.....unsere Kids mögen diese Art des Lernens. ...,2015-06-01,0.0,1.0,False,train,file:///c:/Users/shara/Documents/MasterIS/BD/B...,42949672964,bronze_clean,,2026-03-11 14:16:27.519050+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11994,11942,0007513763,929157076,The Day The Crayons Quit,N,Y,We LOVE this book - it's funny and quirky,"We LOVE this book - it's funny and quirky, a b...",2015-08-06,1.0,0.0,NaN,validation,file:///c:/Users/shara/Documents/MasterIS/BD/B...,1244,bronze_clean,text_cleaned_review_body,2026-03-11 14:16:27.617412+00:00
11995,11962,B00HWGKMJY,543536079,cette nouvelle série est absolument géniale j'...,N,Y,Elementary - Saison 1,Sherlock Holmes revu et corrigé façon moderne ...,2014-05-30,2.0,3.0,NaN,validation,file:///c:/Users/shara/Documents/MasterIS/BD/B...,1245,bronze_clean,,2026-03-11 14:16:27.617412+00:00
11996,11964,B008NW1M4K,691626223,Er ist wieder da: Der Roman,N,Y,Interessant...,"""Der Titel an sich und der Klappentext haben m...",None,NaN,NaN,NaN,validation,file:///c:/Users/shara/Documents/MasterIS/BD/B...,1246,bronze_clean_from_corrupt_record_kept,corrupt_record_unrepaired_kept;invalid_or_miss...,2026-03-11 14:16:27.617412+00:00
11997,11967,B004SYP5WC,758268625,Rappel de factures,N,Y,Ok,"Application de bonne qualité , bonne applicati...",2015-05-26,2.0,1.0,NaN,validation,file:///c:/Users/shara/Documents/MasterIS/BD/B...,1247,bronze_clean,,2026-03-11 14:16:27.617412+00:00


## 9) Export Cleaned Outputs by Split (Parquet)

In [56]:
SILVER_DIR.mkdir(parents=True, exist_ok=True)

for split_name in ['train', 'test', 'validation']:
    out_dir = SILVER_SPLIT_DIRS[split_name]
    out_dir.mkdir(parents=True, exist_ok=True)

    clean_parquet = out_dir / f'cleaned_{split_name}.parquet'
    silver_splits[split_name].to_parquet(clean_parquet, index=False)

    print(f"{split_name:<10} cleaned -> {clean_parquet}")

train      cleaned -> C:\Users\shara\Documents\MasterIS\BD\BigDataProject\data\silver\train\cleaned_train.parquet
test       cleaned -> C:\Users\shara\Documents\MasterIS\BD\BigDataProject\data\silver\test\cleaned_test.parquet
validation cleaned -> C:\Users\shara\Documents\MasterIS\BD\BigDataProject\data\silver\validation\cleaned_validation.parquet
